# 04 Orchestrated run (scheduled by the Fabric Data Pipeline)
Runs all stages in order with logging, retries, the quality gate and reconciliation. Raises an error when the run fails, so the pipeline activity turns red and its failure path (alert email) fires.

In [ ]:
# Parameters (the Data Pipeline overrides these)
load_mode = "full"  # or "incremental"
raw_path = "Files/raw"  # e.g. "Files/increments/2026-11-15" for a change extract
generate_sources = True  # demo only: write synthetic extracts first (no real ERP connected)

In [ ]:
%pip install https://github.com/alexanderburblis/mfg-ops-intelligence/archive/refs/heads/main.zip --quiet

In [ ]:
from mfgops.generate import generate, generate_increment
from mfgops.lakehouse import Lakehouse
from mfgops.orchestrate import run_pipeline

if generate_sources:
    target = "/lakehouse/default/" + raw_path
    print(generate_increment(target) if load_mode == "incremental" else generate(target))
lh = Lakehouse.detect(spark)
lh.raw_path = raw_path
res = run_pipeline(lh, max_error_rate=0.02, mode=load_mode)
print(res.run_id, res.mode, res.status, res.failed_stage, res.message)
if res.status != "success":
    raise RuntimeError(f"Pipeline run {res.run_id} failed at {res.failed_stage}: {res.message}")